# 03 · RMSNorm, rotary position embeddings (RoPE), SwiGLU, grouped-query attention

**Needed for:** LLaMA (LLM track, step 2). RoPE comes back in MLA (notebook 08) and M-RoPE for vision
models (notebook 15).

These four pieces turn the GPT-2 block into a LLaMA block:

| GPT-2 | LLaMA | Why |
|---|---|---|
| LayerNorm | RMSNorm | No mean subtraction, no bias: cheaper, works as well |
| Learned absolute positions | RoPE on q and k | Attention scores depend on *relative* distance; extends to longer contexts |
| GELU MLP (4d) | SwiGLU (about 8d/3, three matrices) | Gated activation trains better at equal parameter count |
| Multi-head attention | Grouped-query attention | Fewer K/V heads shrink the KV cache |

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. RMSNorm

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \odot g$$

Compute in fp32 even when the model runs in bf16 (the mean of squares loses precision otherwise).

In [ ]:
def rmsnorm(x, weight, eps=1e-6):
    xf = x.float()
    out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)
    return (out * weight.float()).type_as(x)


def test_rmsnorm(fn):
    x = torch.randn(2, 5, 16)
    w = torch.randn(16)
    ref = nn.RMSNorm(16, eps=1e-6)
    with torch.no_grad():
        ref.weight.copy_(w)
    torch.testing.assert_close(fn(x, w, eps=1e-6), ref(x))


test_rmsnorm(rmsnorm)
print("matches nn.RMSNorm")

## 2. RoPE

Split each head vector into pairs of dimensions. Pair $j$ at position $m$ is rotated by angle $m\,\theta_j$,
with $\theta_j = \text{base}^{-2j/d}$ (base 10000 in LLaMA). Early pairs rotate fast, later pairs slowly.

Because rotations compose, $\langle R_m q, R_n k\rangle = \langle q, R_{n-m} k\rangle$: the score depends only on
the offset $n-m$. RoPE is applied to **q and k only**, never to v.

There are two common layouts, which give different numbers for the same weights:
- **Interleaved** (Meta's LLaMA code): pairs are dims (0,1), (2,3), ... Treat each pair as a complex number and
  multiply by $e^{i m\theta_j}$.
- **Half-split** (Hugging Face `rotate_half`): pairs are dims (j, j + d/2).

Loading Meta checkpoints into the Hugging Face layout needs a permutation of the q/k projection rows.
Getting this wrong is a classic silent bug: the model runs but outputs garbage.

In [ ]:
def rope_angles(head_dim, seq_len, base=10000.0):
    inv_freq = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)  # (hd/2,)
    return torch.outer(torch.arange(seq_len).float(), inv_freq)                 # (T, hd/2)


def apply_rope_complex(x, angles):
    """Interleaved layout. x: (B, T, H, hd), angles: (T, hd/2)."""
    xc = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))  # (B, T, H, hd/2)
    rot = torch.polar(torch.ones_like(angles), angles)[None, :, None, :]  # e^{i m theta}
    return torch.view_as_real(xc * rot).flatten(-2).type_as(x)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope_half(x, angles):
    """Half-split layout. x: (B, T, H, hd), angles: (T, hd/2)."""
    emb = torch.cat([angles, angles], dim=-1)            # (T, hd)
    cos, sin = emb.cos()[None, :, None, :], emb.sin()[None, :, None, :]
    return x * cos + rotate_half(x) * sin


B, T, H, hd = 2, 12, 4, 16
x = torch.randn(B, T, H, hd)
ang = rope_angles(hd, T)

# the two layouts agree after permuting dims: interleaved pair (2j, 2j+1) <-> half-split pair (j, j + hd/2)
perm = torch.cat([torch.arange(0, hd, 2), torch.arange(1, hd, 2)])
torch.testing.assert_close(apply_rope_half(x[..., perm], ang), apply_rope_complex(x, ang)[..., perm])
print("layouts agree up to a permutation")

In [ ]:
def test_rope_relative(fn):
    """The score between positions m and n must depend only on n - m."""
    q, k = torch.randn(1, 1, 1, hd), torch.randn(1, 1, 1, hd)
    ang_long = rope_angles(hd, 64)

    def score(m, n):
        qm = fn(q, ang_long[m:m + 1])
        kn = fn(k, ang_long[n:n + 1])
        return (qm * kn).sum()

    torch.testing.assert_close(score(3, 10), score(30, 37))
    torch.testing.assert_close(score(0, 0), (q * k).sum())  # position 0 is no rotation


test_rope_relative(apply_rope_complex)
test_rope_relative(apply_rope_half)
print("scores depend only on relative position")

## 3. SwiGLU feed-forward

$$\text{FFN}(x) = W_2\big(\text{SiLU}(W_1 x) \odot W_3 x\big)$$

Three matrices instead of two, so LLaMA shrinks the hidden size to about $\frac{2}{3}\cdot 4d$ (rounded up to a
multiple of 256) to keep the parameter count equal to a 4d GELU MLP.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim, multiple_of=256):
        super().__init__()
        hidden = int(2 * 4 * dim / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)
        self.w1 = nn.Linear(dim, hidden, bias=False)  # gate
        self.w3 = nn.Linear(dim, hidden, bias=False)  # up
        self.w2 = nn.Linear(hidden, dim, bias=False)  # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


ffn = SwiGLU(4096)
print("LLaMA-7B hidden size:", ffn.w1.out_features)  # 11008 in the paper
gelu_params = 2 * 4096 * 4 * 4096
print("SwiGLU params / GELU-MLP params:", round(sum(p.numel() for p in ffn.parameters()) / gelu_params, 3))

## 4. Grouped-query attention

$H_q$ query heads share $H_{kv}$ key/value heads ($H_q$ divisible by $H_{kv}$). Each group of $H_q / H_{kv}$
query heads reads the same K and V. $H_{kv} = 1$ is multi-query attention; $H_{kv} = H_q$ is ordinary MHA.
The KV cache shrinks by a factor of $H_q / H_{kv}$.

In [ ]:
def repeat_kv(x, n_rep):
    """(B, H_kv, T, hd) -> (B, H_kv * n_rep, T, hd); kv head i serves query heads i*n_rep ... i*n_rep + n_rep - 1."""
    if n_rep == 1:
        return x
    B, Hkv, T, hd = x.shape
    return x[:, :, None].expand(B, Hkv, n_rep, T, hd).reshape(B, Hkv * n_rep, T, hd)


def gqa_attention(q, k, v, causal=True):
    """q: (B, Hq, T, hd); k, v: (B, Hkv, T, hd)."""
    n_rep = q.shape[1] // k.shape[1]
    k, v = repeat_kv(k, n_rep), repeat_kv(v, n_rep)
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal)


def test_repeat_kv(fn):
    kv = torch.randn(2, 2, 5, 8)
    out = fn(kv, 3)
    assert out.shape == (2, 6, 5, 8)
    torch.testing.assert_close(out, kv.repeat_interleave(3, dim=1))


test_repeat_kv(repeat_kv)
q, k, v = torch.randn(2, 8, 10, 16), torch.randn(2, 2, 10, 16), torch.randn(2, 2, 10, 16)
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)
torch.testing.assert_close(gqa_attention(q, k, v), ref)
print("matches SDPA(enable_gqa=True)")

## 5. KV cache size, which is why GQA exists

Per token, per layer: $2 \times H_{kv} \times d_{head} \times \text{bytes}$.

In [ ]:
def kv_cache_gb(n_layers, n_kv_heads, head_dim, seq_len, batch, bytes_per=2):
    return 2 * n_layers * n_kv_heads * head_dim * seq_len * batch * bytes_per / 1e9


print("LLaMA-2-70B as MHA (64 kv heads), 4k ctx, batch 8:", round(kv_cache_gb(80, 64, 128, 4096, 8), 1), "GB")
print("LLaMA-2-70B GQA    ( 8 kv heads), 4k ctx, batch 8:", round(kv_cache_gb(80, 8, 128, 4096, 8), 1), "GB")

## Things to be able to explain

- Why RoPE on q and k but not v?
- Why does a model trained at 4k context degrade at 32k, and what do position interpolation / NTK scaling /
  YaRN change? (They rescale the angles so long positions look like trained ones.)
- Where does the norm go in a pre-norm block, and why does pre-norm train more stably than post-norm?

## Drill

In [ ]:
def my_rmsnorm(x, weight, eps=1e-6):
    raise NotImplementedError


def my_apply_rope(x, angles):
    """Either layout. x: (B, T, H, hd), angles: (T, hd/2)."""
    raise NotImplementedError


def my_repeat_kv(x, n_rep):
    raise NotImplementedError


drill(test_rmsnorm, my_rmsnorm)
drill(test_rope_relative, my_apply_rope)
drill(test_repeat_kv, my_repeat_kv)